In [85]:
!pip install -U altair vega_datasets

In [86]:
import altair as alt
import pandas as pd

df = pd.read_excel("/content/military_spending_data.xlsx")

In [87]:
df.head()

,Country,1949,1950,1951,1952,1953,1954,1955,1956,1957,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,United States of America,181095.5,194625.0,416423.8,586599.3,603952.7,521100.6,493268.1,499722.0,513981.1,...,836290.9,827674.3,852585.0,901032.1,943446.6,933334.5,922552.5,943032.8,1004901.0,929162.9
1,China,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,207375.4,220129.3,233023.7,244376.0,255894.6,262558.7,274158.2,292220.8,311900.4,335017.6
2,Russia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,80545.6,65267.8,62808.7,65623.9,67165.7,68496.3,88678.7,108959.5,149401.9,158236.4
3,Germany,0.0,0.0,0.0,0.0,19577.7,19821.4,22903.0,21812.4,26546.3,...,48902.7,50305.3,51693.6,56704.7,60418.4,59887.7,62499.5,70204.6,86148.3,106727.2
4,India,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3046.0,3642.9,...,67201.7,71935.6,74468.9,79726.1,80209.7,79701.3,83218.2,85232.5,85597.1,93254.7


In [88]:
# Google AI-Assisted Dataframe Cleaning.

# Only get the Top 6 countries with the highest Military Budgets.
# Commented out Code was removed because I realized "Rest of World" was being artificially deflated.
df = df.iloc[0:6].copy()
#df_rest = df.iloc[6:].copy()

# Aggregate the rest into a single "Rest of World" row
# This sums up the numeric values for every year column (1949 to 2025)
#year_columns = [year for year in range(1949, 2026)]
#rest_summed = df_rest[year_columns].sum().to_frame().T
#rest_summed["Country"] = "Rest of World"

# Move "Country" to the leftmost position
#cols = ["Country"] + [col for col in rest_summed.columns if col != "Country"]
#rest_summed = rest_summed[cols]

# Combine Top 6 with the new aggregated row.
#df = pd.concat([df_top6, rest_summed], ignore_index=True)

# Melt the DataFrame from wide to long format.
# This turns columns "1949", "1950", etc. into a "Year" column and a "Spending" column.
df = pd.melt(
    df,
    id_vars=["Country"],
    var_name="Year",
    value_name="Military_Spending",
)

# Severely cuts down the number of rows. Compare it to the original graph.
df = df[df["Year"] >= 1990]

# Convert Year to datetime for Altair
df["Year"] = pd.to_datetime(df["Year"], format="%Y")

df

,Country,Year,Military_Spending
246,United States of America,1990-01-01,780581.1
247,China,1990-01-01,21639.6
248,Russia,1990-01-01,0.0
249,Germany,1990-01-01,70619.4
250,India,1990-01-01,21862.1
...,...,...,...
457,China,2025-01-01,335017.6
458,Russia,2025-01-01,158236.4
459,Germany,2025-01-01,106727.2
460,India,2025-01-01,93254.7


In [89]:
# Find the final year's data to determine the legend sorting order
# This grabs the last year (2025) and sorts countries by spending descending
final_year_df = df[df["Year"] == df["Year"].max()].sort_values(
    by="Military_Spending", ascending=False
)
legend_order = final_year_df["Country"].tolist()

chart5 = (
    alt.Chart(df)
    .mark_line()
    .encode(
        x=alt.X("Year:T", title="Year"),
        y=alt.Y("Military_Spending:Q", title="Military Spending (Constant 2024 $Millions USD)"),
        color=alt.Color("Country:N",
                        title="Country",
                        sort=legend_order,
                        scale=alt.Scale(scheme="category20")
        ),
        tooltip=["Country:N",
                 alt.Tooltip("Year:T", format="%Y", title="Year"),
                 alt.Tooltip("Military_Spending:Q", format="$,.0f", title="Spending (2024 $M)")
        ],
    )
    .properties(width=500,
                height=360,
                title="Military Spending Over Time (1990-2025) (Constant 2024 $Millions USD)",
                )
    .interactive()
)

chart5

alt.Chart(...)

In [90]:
# Pie Chart for 2025 Military Spending.
# Ultimately did not decide to use this.

# Filter the data for the year 2025 only.
df_2025 = df[df["Year"].dt.year == 2025]

# Establish the legend and slice order based on spending.
pie_order = df_2025.sort_values(by="Military_Spending", ascending=False)["Country"].tolist()

# Build the Pie Chart.
chart6 = (
    alt.Chart(df_2025)
    .mark_arc(outerRadius=120)  # Controls the size of the pie
    .encode(
        theta=alt.Theta("Military_Spending:Q", title="Military Spending"),
        color=alt.Color(
            "Country:N",
            title="Country",
            sort=pie_order,  # Sorts legend from largest to smallest slice
            scale=alt.Scale(scheme="category20"),
        ),
        order=alt.Order("Military_Spending:Q", sort="descending"),  # Arranges slices in order
        tooltip=[
            "Country:N",
            alt.Tooltip("Military_Spending:Q", format="$,.0f", title="Spending ($M)"),
        ],
    )
    .properties(
        width=400,
        height=400,
        title="Share of Global Military Spending (2025)",
    )
)

chart6


alt.Chart(...)

In [92]:
# Exclude the United States to see the other countries more clearly.

df = df[df["Country"] != "United States of America"]

# Find the final year's data to determine the legend sorting order
# This grabs the last year (2025) and sorts countries by spending descending
final_year_df = df[df["Year"] == df["Year"].max()].sort_values(
    by="Military_Spending", ascending=False
)
legend_order = final_year_df["Country"].tolist()

chart6 = (
    alt.Chart(df)
    .mark_line()
    .encode(
        x=alt.X("Year:T", title="Year"),
        y=alt.Y("Military_Spending:Q", title="Military Spending (Constant 2024 $Millions USD)"),
        color=alt.Color("Country:N",
                        title="Country",
                        sort=legend_order,
                        scale=alt.Scale(scheme="category20")
        ),
        tooltip=["Country:N",
                 alt.Tooltip("Year:T", format="%Y", title="Year"),
                 alt.Tooltip("Military_Spending:Q", format="$,.0f", title="Spending (2024 $M)")
        ],
    )
    .properties(width=500,
                height=360,
                title="Military Spending Over Time (1990-2025) (Constant 2024 $Millions USD) Excluding the USA",
                )
    .interactive()
)

chart6

alt.Chart(...)